In [1]:
export MODEL="unsloth/Qwen3.5-27B-GGUF:Q3_K_M"
# export MODEL="unsloth/gemma-4-E4B-it-GGUF:UD-Q6_K_XL"
# to set sudo sysctl iogpu.wired_limit_mb=14800. Needs to be set on termimal outside for pure GPU
sysctl iogpu.wired_limit_mb # to check

iogpu.wired_limit_mb: 15000


In [2]:
llama-fit-params -hf $MODEL # This is just for downloading and some initial values to probe

load_backend: loaded BLAS backend from /opt/homebrew/Cellar/ggml/0.9.11/libexec/libggml-blas.so
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.022 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSet

In [2]:
# Configurable batch sizes to test to fit context
# Though it looks like batch sizes have no effects 
# as per README 
# "Increasing this value above the value of the physical batch size may improve prompt processing performance 
# when using multiple GPUs with pipeline parallelism. Default: `2048`" 

BATCH_SIZES="1024 2048" # 2 args so we can see the diff, shoud not make a difference  
UBATCH_SIZES="64 128 256 512"
FITT="256 512"

echo "Testing different batch/ubatch/fitt combinations for ${MODEL}..." 
  
for ub in $UBATCH_SIZES; do  
    for b in $BATCH_SIZES; do
        for ft in $FITT; do
            # Get fitted parameters  
            #OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub | grep -o '\-c [0-9]* \-ngl [0-9]*')  \
            
            # need to add fitt for nvidia
            OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub -fitt $ft 2>&1)
            
            #echo "Raw output: $OUTPUT"  # Debug line  
            
            # Extract context and GPU layers more robustly  
            CTX=$(echo "$OUTPUT" | grep -o '\-c [0-9]*' | awk '{print $2}')  
            NGL=$(echo "$OUTPUT" | grep -o '\-ngl -\?[0-9]*' | awk '{print $2}')  
            
            if [ ! -z "$CTX" ] && [ ! -z "$NGL" ]; then  
                echo "ub: $ub, b: $b, fitt: $ft, ctx: $CTX, ngl: $NGL"  
            else  
                echo "No valid parameters found"  
            fi 
        done
    done  
done 

Testing different batch/ubatch/fitt combinations for unsloth/Qwen3.5-27B-GGUF:Q3_K_M...
ub: 64, b: 1024, fitt: 256, ctx: 32256, ngl: -1
ub: 64, b: 1024, fitt: 512, ctx: 28160, ngl: -1
ub: 64, b: 2048, fitt: 256, ctx: 32256, ngl: -1
ub: 64, b: 2048, fitt: 512, ctx: 28160, ngl: -1
ub: 128, b: 1024, fitt: 256, ctx: 31232, ngl: -1
ub: 128, b: 1024, fitt: 512, ctx: 27136, ngl: -1
ub: 128, b: 2048, fitt: 256, ctx: 31232, ngl: -1
ub: 128, b: 2048, fitt: 512, ctx: 27136, ngl: -1
ub: 256, b: 1024, fitt: 256, ctx: 29184, ngl: -1
ub: 256, b: 1024, fitt: 512, ctx: 25088, ngl: -1
ub: 256, b: 2048, fitt: 256, ctx: 29184, ngl: -1
ub: 256, b: 2048, fitt: 512, ctx: 25088, ngl: -1
ub: 512, b: 1024, fitt: 256, ctx: 25088, ngl: -1
ub: 512, b: 1024, fitt: 512, ctx: 20992, ngl: -1
ub: 512, b: 2048, fitt: 256, ctx: 25088, ngl: -1
ub: 512, b: 2048, fitt: 512, ctx: 20992, ngl: -1


In [3]:
# Pure GPU
llama-bench -hf $MODEL -ngl 99 -t 8 -ub 128,256,512 -p 1000,2000 -n 256,512

load_backend: loaded BLAS backend from /opt/homebrew/Cellar/ggml/0.9.11/libexec/libggml-blas.so
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.009 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSet

In [ ]:
# From llama.cpp readme. 
# -b 
# "Increasing this value above the value of the physical batch size may improve prompt processing performance 
# when using multiple GPUs with pipeline parallelism. Default: `2048`" 
# so may not be applicable to mac mini

In [ ]:
# batched bench seems the closer to llama-server as judged from the logs, thats' why it is here
# llama-batched-bench -hf $MODEL \
#     -ngl 99 --threads 8 -fa on \
#     -c 16000 \
#     -npp 1000 -ntg 1000 -npl 1

In [4]:
llama-fit-params -hf $MODEL -fitt 512 # fitt leaves that around of memory in MiB in the target devices

load_backend: loaded BLAS backend from /opt/homebrew/Cellar/ggml/0.9.11/libexec/libggml-blas.so
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.023 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSet

In [9]:
llama-server -hf $MODEL \
    --threads-http 1 --fit on --threads 8 --no-mmproj --reasoning off \
    --no-warmup -ub 64 -ngl -1 -np 1 --host 0.0.0.0 # setting ub to 64 did not help

load_backend: loaded BLAS backend from /opt/homebrew/Cellar/ggml/0.9.11/libexec/libggml-blas.so
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.009 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSet